### Utilizar y cargar las librerías necesarías para el EDA

In [1]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, '../src')
from extraccion import Extraccion

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

print('Librerías cargadas correctamente.')

KeyboardInterrupt: 

#### Iniciar la conexión a MongoDB

In [ ]:
ext = Extraccion(uri='mongodb://localhost:27017', db_name='airbnb_argentina')
ext.conectar()
dfs = ext.extraer_todo()
ext.cerrar_conexion()

df_listings  = dfs['listings']
df_calendar  = dfs['calendar']
df_reviews   = dfs['reviews']

print(f'Listings:  {df_listings.shape[0]:,} filas × {df_listings.shape[1]} columnas')
print(f'Calendar:  {df_calendar.shape[0]:,} filas × {df_calendar.shape[1]} columnas')
print(f'Reviews:   {df_reviews.shape[0]:,} filas × {df_reviews.shape[1]} columnas')

# Punto 2: Analisis exploratorio de datos (EDA)

## Item 2.1. Entendimiento general de los datos

### Para cada colección:

#### Mostrar las primeras filas

Para la colección de Listings

In [ ]:
print('=== Primeras 5 filas de Listings ===')
df_listings.head(5)

In [ ]:
print(f'Shape: {df_listings.shape}')
df_listings.info()

In [ ]:
# Descripción estadística de variables numéricas relevantes
cols_num = ['price', 'minimum_nights', 'maximum_nights', 'availability_365',
            'number_of_reviews', 'review_scores_rating', 'reviews_per_month']
cols_existentes = [c for c in cols_num if c in df_listings.columns]
df_listings[cols_existentes].describe().round(2)

Para la colección de Calendar

In [ ]:
df_calendar.head(5)

In [ ]:
print(f'Shape: {df_calendar.shape}')
df_calendar.info()

Para la colección de Reviews

In [ ]:
df_reviews.head(5)

In [ ]:
print(f'Shape: {df_reviews.shape}')
df_reviews.info()

## 2.2. Calidad de los datos

#### Identificación de valores nulos

In [ ]:
def resumen_nulos(df, nombre):
    """Genera un resumen de nulos ordenado por porcentaje."""
    nulos = df.isnull().sum()
    porcentaje = (nulos / len(df) * 100).round(2)
    resumen = pd.DataFrame({'nulos': nulos, 'porcentaje': porcentaje})
    resumen = resumen[resumen['nulos'] > 0].sort_values('porcentaje', ascending=False)
    print(f'\n=== Nulos en {nombre} ({len(resumen)} columnas con nulos) ===')
    display(resumen.head(20))
    return resumen

nulos_listings = resumen_nulos(df_listings, 'Listings')
nulos_calendar = resumen_nulos(df_calendar, 'Calendar')
nulos_reviews  = resumen_nulos(df_reviews, 'Reviews')

In [ ]:
print('=== Filas con null ===')
df_listings[df_listings['has_availability'].isnull()].head(13)

In [ ]:
# Mapa de calor de nulos — Listings (top 20 columnas con más nulos)
top_nulos = nulos_listings.head(20).index.tolist()
if top_nulos:
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(
        df_listings[top_nulos].isnull().astype(int).head(500),
        cmap='Reds', cbar=False, ax=ax
    )
    ax.set_title('Mapa de valores nulos — Listings (primeras 500 filas, top 20 columnas)')
    ax.set_xlabel('Columnas')
    ax.set_ylabel('Filas')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../data/grafica_nulos_listings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gráfica guardada.')

#### Registros duplicados

In [ ]:
def df_hashable(df):
    """Convierte columnas con listas u objetos no hasheables a string para poder comparar duplicados."""
    df_temp = df.copy()
    for col in df_temp.columns:
        if df_temp[col].dtype == object:
            df_temp[col] = df_temp[col].apply(
                lambda x: str(x) if isinstance(x, (list, dict)) else x
            )
    return df_temp

for nombre, df in [('Listings', df_listings), ('Calendar', df_calendar), ('Reviews', df_reviews)]:
    df_safe = df_hashable(df)
    
    dup_total = df_safe.duplicated().sum()
    
    col_id = 'id' if 'id' in df_safe.columns else None
    if col_id:
        dup_id = df_safe[col_id].astype(str).duplicated().sum()
    else:
        dup_id = 'N/A'
    
    print(f'{nombre}: {dup_total} duplicados totales | {dup_id} duplicados por ID')

#### Valores atípicos detectados

In [ ]:
# Normalizar precio para análisis (eliminar $ y comas)
if 'price' in df_listings.columns:
    precio_num = (
        df_listings['price']
        .astype(str)
        .str.replace(r'[\$,\s]', '', regex=True)
        .replace('nan', None)
    )
    precio_num = pd.to_numeric(precio_num, errors='coerce')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Histograma precio
    precio_filtrado = precio_num[precio_num < precio_num.quantile(0.95)]
    axes[0].hist(precio_filtrado.dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[0].set_title('Distribución de Precio (sin top 5%)')
    axes[0].set_xlabel('Precio')
    axes[0].set_ylabel('Frecuencia')

    # Boxplot precio
    axes[1].boxplot(precio_num.dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[1].set_title('Boxplot de Precio')
    axes[1].set_ylabel('Precio')

    # Boxplot minimum_nights
    if 'minimum_nights' in df_listings.columns:
        mn = pd.to_numeric(df_listings['minimum_nights'], errors='coerce')
        mn_filtrado = mn[mn < mn.quantile(0.99)]
        axes[2].boxplot(mn_filtrado.dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor='lightgreen'))
        axes[2].set_title('Boxplot de Minimum Nights')
        axes[2].set_ylabel('Noches mínimas')

    plt.suptitle('Outliers en variables numéricas clave', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig('../data/grafica_outliers.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Estadísticas de outliers
    Q1, Q3 = precio_num.quantile(0.25), precio_num.quantile(0.75)
    IQR = Q3 - Q1
    outliers = precio_num[(precio_num < Q1 - 1.5*IQR) | (precio_num > Q3 + 1.5*IQR)]
    print(f'\nOutliers en precio (método IQR): {len(outliers):,} registros ({len(outliers)/len(precio_num)*100:.1f}%)')
    print(f'Precio mediano: ${precio_num.median():.2f}')
    print(f'Precio máximo: ${precio_num.max():.2f}')

### Distribución de variables categóricas

#### Tipo de habitación

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
if 'room_type' in df_listings.columns:
    conteo = df_listings['room_type'].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Barras
    conteo.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted'), edgecolor='white')
    axes[0].set_title('Conteo por tipo de habitación')
    axes[0].set_xlabel('Tipo')
    axes[0].set_ylabel('Cantidad')
    axes[0].tick_params(axis='x', rotation=30)
    for bar in axes[0].patches:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                     f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)
    
    # Torta
    axes[1].pie(conteo, labels=conteo.index, autopct='%1.1f%%',
                colors=sns.color_palette('muted'), startangle=90)
    axes[1].set_title('Proporción por tipo de habitación')
    
    plt.tight_layout()
    plt.savefig('../data/grafica_room_type.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(conteo.to_string())

#### Ver los datos con mas listings

In [ ]:
col_barrio = next((c for c in ['neighbourhood_cleansed', 'neighbourhood', 'neighborhood']
                   if c in df_listings.columns), None)

if col_barrio:
    top_barrios = df_listings[col_barrio].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    top_barrios.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 15 barrios con más listings')
    ax.set_xlabel('Cantidad de listings')
    ax.set_ylabel('Barrio')
    for i, v in enumerate(top_barrios.sort_values()):
        ax.text(v + 10, i, f'{v:,}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../data/grafica_barrios.png', dpi=150, bbox_inches='tight')
    plt.show()

#### Disponibilidad promedio - Noches minimas y maximas - Ocupacion estimada

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('Análisis de Disponibilidad', fontsize=16, fontweight='bold')

# 1. Disponibilidad promedio por rango
disponibilidad_media = df_listings[['availability_30', 'availability_60', 'availability_90', 'availability_365']].mean().round(2)
axes[0, 0].bar(disponibilidad_media.index, disponibilidad_media.values, color='steelblue')
axes[0, 0].set_title('Disponibilidad promedio por rango')
axes[0, 0].set_ylabel('Días promedio')
axes[0, 0].tick_params(axis='x', rotation=15)

# 2. Distribución ocupación estimada
axes[0, 1].hist(df_listings['estimated_occupancy_l365d'].dropna(), bins=30, color='coral', edgecolor='white')
axes[0, 1].set_title('Distribución ocupación estimada (365d)')
axes[0, 1].set_xlabel('Días ocupados')
axes[0, 1].set_ylabel('Cantidad de propiedades')

# 3. Distribución noches mínimas (sin outliers)
min_nights = df_listings['minimum_nights'].clip(upper=30)
axes[1, 0].hist(min_nights, bins=30, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Noches mínimas (hasta 30)')
axes[1, 0].set_xlabel('Noches')
axes[1, 0].set_ylabel('Cantidad de propiedades')

# 4. Distribución noches máximas (sin outliers)
max_nights = df_listings['maximum_nights'].clip(upper=365)
axes[1, 1].hist(max_nights, bins=30, color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('Noches máximas (hasta 365)')
axes[1, 1].set_xlabel('Noches')
axes[1, 1].set_ylabel('Cantidad de propiedades')

# 5. Boxplot disponibilidad
df_listings[['availability_30', 'availability_60', 'availability_90', 'availability_365']].plot(
    kind='box', ax=axes[2, 0], color='steelblue'
)
axes[2, 0].set_title('Boxplot disponibilidad por rango')
axes[2, 0].set_ylabel('Días')
axes[2, 0].tick_params(axis='x', rotation=15)

# 6. Boxplot noches mínimas y máximas (sin outliers)
df_listings[['minimum_nights', 'maximum_nights']].clip(upper=365).plot(
    kind='box', ax=axes[2, 1], color='coral'
)
axes[2, 1].set_title('Boxplot noches mínimas y máximas')
axes[2, 1].set_ylabel('Noches')

plt.tight_layout()
plt.savefig('../data/Disponibilidad promedio.png', dpi=150, bbox_inches='tight')
plt.show()

#### Precio mediano por tipo de habitación

In [ ]:
"""if 'room_type' in df_listings.columns and 'price' in df_listings.columns:
    df_temp = df_listings.copy()
    df_temp['price_num'] = pd.to_numeric(
        df_temp['price'].astype(str).str.replace(r'[\$,\s]', '', regex=True),
        errors='coerce'
    )
    precio_por_tipo = df_temp.groupby('room_type')['price_num'].median().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    precio_por_tipo.plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
    ax.set_title('Precio mediano por tipo de habitación')
    ax.set_xlabel('Tipo de habitación')
    ax.set_ylabel('Precio mediano')
    ax.tick_params(axis='x', rotation=30)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig('../data/grafica_precio_room_type.png', dpi=150, bbox_inches='tight')
    plt.show()"""

#### Disponibilidad a lo largo del año 

In [ ]:
if 'available' in df_calendar.columns and 'date' in df_calendar.columns:
    df_cal = df_calendar.copy()
    df_cal['date'] = pd.to_datetime(df_cal['date'], errors='coerce')
    df_cal['mes'] = df_cal['date'].dt.month
    df_cal['disponible'] = df_cal['available'].map({'t': 1, 'f': 0, True: 1, False: 0})
    
    disponibilidad_mes = df_cal.groupby('mes')['disponible'].mean() * 100
    
    meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
    disponibilidad_mes.index = [meses[i-1] for i in disponibilidad_mes.index]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    disponibilidad_mes.plot(kind='line', marker='o', ax=ax, color='teal', linewidth=2)
    ax.fill_between(range(len(disponibilidad_mes)), disponibilidad_mes.values, alpha=0.2, color='teal')
    ax.set_xticks(range(len(disponibilidad_mes)))
    ax.set_xticklabels(disponibilidad_mes.index)
    ax.set_title('Disponibilidad promedio mensual (%)')
    ax.set_ylabel('% disponibilidad')
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    plt.tight_layout()
    plt.savefig('../data/grafica_disponibilidad_mensual.png', dpi=150, bbox_inches='tight')
    plt.show()

#### Evolución de reviews por año

In [ ]:
if 'date' in df_reviews.columns:
    df_rev = df_reviews.copy()
    df_rev['date'] = pd.to_datetime(df_rev['date'], errors='coerce')
    df_rev['anio'] = df_rev['date'].dt.year
    
    reviews_anio = df_rev['anio'].value_counts().sort_index()
    reviews_anio = reviews_anio[reviews_anio.index >= 2015]  # Filtrar años relevantes
    
    fig, ax = plt.subplots(figsize=(12, 5))
    reviews_anio.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_title('Cantidad de reviews por año')
    ax.set_xlabel('Año')
    ax.set_ylabel('Cantidad de reviews')
    ax.tick_params(axis='x', rotation=0)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig('../data/grafica_reviews_anio.png', dpi=150, bbox_inches='tight')
    plt.show()

### Correlaciones entre variables numéricas

In [ ]:
cols_correlacion = ['minimum_nights', 'availability_365', 'number_of_reviews',
                    'review_scores_rating', 'reviews_per_month', 'calculated_host_listings_count']
cols_existentes = [c for c in cols_correlacion if c in df_listings.columns]

if len(cols_existentes) >= 3:
    df_corr = df_listings[cols_existentes].apply(pd.to_numeric, errors='coerce')
    corr_matrix = df_corr.corr()
    
    fig, ax = plt.subplots(figsize=(10, 7))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, ax=ax,
        square=True, linewidths=0.5
    )
    ax.set_title('Matriz de correlación — Variables numéricas Listings')
    plt.tight_layout()
    plt.savefig('../data/grafica_correlacion.png', dpi=150, bbox_inches='tight')
    plt.show()

### Analisis de transformaciones necesarias 

Campo amenities de la colección listings

In [ ]:
if 'amenities' in df_listings.columns:
    muestra = df_listings['amenities'].dropna().head(3)
    for i, val in enumerate(muestra):
        print(f'Registro {i+1}: {str(val)[:200]}')
    print('\n→ El campo amenities es una lista en formato string. Se debe parsear con ast.literal_eval.')

Campo price en la colección listings

In [ ]:
"""if 'price' in df_listings.columns:
    print('Tipo actual:', df_listings['price'].dtype)
    print('Ejemplos:', df_listings['price'].dropna().head(5).tolist())
    print('\n→ Requiere eliminar $, comas y convertir a float.')"""

Campo date en la coleccion calendar

In [ ]:
if 'date' in df_calendar.columns:
    print('Tipo actual:', df_calendar['date'].dtype)
    print('Ejemplos:', df_calendar['date'].dropna().head(5).tolist())
    print('\n→ Se convertirá a datetime y se derivarán: año, mes, día, trimestre.')